# AI Final Project: Flower Image Classification using SVM and CNN
By: Sarah Ogden and Elaina Hall

# Import Data

In [1]:
import os
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from PIL import Image

# Define paths
TRAIN_DIR = 'dataset_split/train'
TEST_DIR = 'dataset_split/test'

# Count images per class
def count_images(root_dir):
    counts = defaultdict(int)
    for class_dir in Path(root_dir).iterdir():
        if class_dir.is_dir():
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
            counts[class_dir.name] = len(images)
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("Training set - Images per class:")
for cls in sorted(train_counts.keys()):
    print(f"  {cls}: {train_counts[cls]}")
print(f"\nTotal train: {sum(train_counts.values())}")

print("\nTest set - Images per class:")
for cls in sorted(test_counts.keys()):
    print(f"  {cls}: {test_counts[cls]}")
print(f"\nTotal test: {sum(test_counts.values())}")

# Verify image sizes
print("\nSample image dimensions:")
for cls in list(train_counts.keys())[:3]:
    img_files = list((Path(TRAIN_DIR) / cls).glob('*.*'))
    if img_files:
        img = Image.open(img_files[0])
        print(f"  {cls}: {img.size}")

ModuleNotFoundError: No module named 'matplotlib'

# SVM Model

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import cv2

# Load and preprocess images into feature vectors
def load_images_for_svm(root_dir, target_size=(224, 224)):
    X, y = [], []
    classes = sorted([d.name for d in Path(root_dir).iterdir() if d.is_dir()])
    
    for class_idx, cls in enumerate(classes):
        img_files = list((Path(root_dir) / cls).glob('*.*'))
        for img_path in img_files:
            try:
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.resize(img, target_size)
                    X.append(img.flatten())  # Flatten to 1D vector
                    y.append(class_idx)
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    
    return np.array(X), np.array(y), classes

print("Loading training images for SVM...")
X_train, y_train, classes = load_images_for_svm(TRAIN_DIR)
print(f"Training data shape: {X_train.shape}")

print("Loading test images for SVM...")
X_test, y_test, _ = load_images_for_svm(TEST_DIR)
print(f"Test data shape: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM
print("\nTraining SVM (this may take a few minutes)...")
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', verbose=1)
svm_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_svm = svm_model.predict(X_test_scaled)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
print(f"\nSVM Accuracy: {svm_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=classes))

# CNN Model

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

# Use data generators for efficient loading
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Build CNN with transfer learning
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
out = layers.Dense(train_generator.num_classes, activation='softmax')(x)

cnn_model = models.Model(base_model.input, out)
cnn_model.compile(optimizer=optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Train CNN
print("Training CNN...")
history = cnn_model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,
    verbose=1
)

# Evaluate on test set
cnn_loss, cnn_accuracy = cnn_model.evaluate(test_generator)
print(f"\nCNN Accuracy: {cnn_accuracy:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('CNN Training History')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('CNN Loss History')
plt.tight_layout()
plt.show()